In [ ]:
DEBUG = False

In [ ]:

import glob
import subprocess
from structural_similarity import load_results, build_inner_args
import yaml
import glob
import subprocess
import os
from argparse import Namespace
BASE_MODEL =  "Olmo-3-1025-7B"# "Qwen3-0.6B-Base"
MODELS = [
    BASE_MODEL + sft for sft in [
    "_ftrace_date-lm_conversational_lr0.0001_seed42",
    "_tulu-3-sft-olmo-2-mixture-0225_lr0.0001_seed42",
    "_ftrace_date-lm_conversational_lr0.0001_seed42",
    "_tulu-3-sft-personas-code_lr0.0001_seed42",
    "_tulu-3-sft-personas-math_lr0.0001_seed42"]
]
with open("experiments.yaml", "r") as f:
    cfg = yaml.safe_load(f)

base_models = [b.split('/')[1] for b in cfg["base_models"]]
experiments_ = cfg["experiments"]

experiments = []
for base in base_models:
    for exp in experiments_:
        new_exp = exp.copy()
        new_exp["model"] = f"./models/{base}{exp['model']}"
        if not os.path.exists(new_exp["model"]):
            print(f"[ERROR] Model {new_exp['model']} not found")
            exit(-1)
        experiments.append(new_exp)


scripts = [
    "representations/run_gradsim_normalized.py",
    "representations/run_bm25.py",
    "representations/run_embd_llama-embed-nemotron-8b.py"
]
from collections import defaultdict 
data = defaultdict(list)
for script_path in scripts:
    for exp in experiments:
        if not any([model in exp["model"] for model in MODELS]):
            continue

        args_dict = {
            "model": exp["model"],
            "train_dataset_name": exp["dataset"],
            "train_dataset_split": "test",
            "test_dataset_name": exp["dataset"],
            "test_dataset_split": "test",
        }

        args = Namespace(**args_dict)

        module_name = script_path.replace(".py", "").replace("/", ".")

      
        
        
        inner_args = build_inner_args(args, func_only=False)
        # print(inner_args)
        representation_builder = lambda module_name=module_name, inner_args=inner_args: load_results(
            module_name,
            inner_args
        )[0]
        data[exp["model"]].append((module_name, args,representation_builder))
        # subprocess.run(cmd)
    


In [ ]:

import matplotlib.colors as mcolors

base_colors = tuple(
    mcolors.to_rgb(tuple(c/255 for c in rgb))
    for rgb in [
        (86, 91, 96),
        (154, 215, 246),
        (250, 192, 154),
        (192, 222, 186),
        (247, 190, 209),
        (255, 243, 109)
    ]
)
from matplotlib import pyplot as plt
import matplotlib.patches as patches

fig, ax = plt.subplots()

ax.set_xlim(0, len(base_colors))
ax.set_ylim(0, 1)

for i, c in enumerate(base_colors):
    rect = patches.Rectangle((i, 0), 1, 1, facecolor=c)
    ax.add_patch(rect)

ax.set_xticks([])
ax.set_yticks([])
ax.set_frame_on(False)

plt.show()

In [ ]:
import torch
import numpy as np

EPS = 1e-8


def standardize(x):
    x = np.array(x)
    return (x - x.mean()) / (x.std() + EPS)


def analyze_representations(representations, dataset, k=10):
    reps = {name: tensor.float() for name, _, tensor in representations}
    names = list(reps.keys())
    knn_maps = {}
    results = {}
    n = len(dataset)

    for i in range(len(names)):
        for j in range(i + 1, len(names)):

            r_i, r_j = names[i], names[j]

            X_i = reps[r_i]
            X_j = reps[r_j]

            # ------------------------------------------------------------
            # STEP 1: compute full similarity matrices ONCE
            # ------------------------------------------------------------
            S_i = X_i @ X_i.T   # (n, n)
            S_j = X_j @ X_j.T   # (n, n)

            # ------------------------------------------------------------
            # STEP 2: remove self-match (vectorized)
            # ------------------------------------------------------------
            idx = torch.arange(n)
            S_i[idx, idx] = -1e9
            S_j[idx, idx] = -1e9

            # ------------------------------------------------------------
            # STEP 3: top-k for ALL queries at once
            # ------------------------------------------------------------
            knn_i = torch.topk(S_i, k=k, dim=1).indices  # (n, k)
            knn_j = torch.topk(S_j, k=k, dim=1).indices
            knn_maps[r_i] = knn_i
            knn_maps[r_j] = knn_j
            # ------------------------------------------------------------
            # STEP 4: vectorized overlap
            # ------------------------------------------------------------
            overlaps = []

            knn_i_np = knn_i.cpu().numpy()
            knn_j_np = knn_j.cpu().numpy()

            for q in range(n):
                overlaps.append(
                    len(set(knn_i_np[q]) & set(knn_j_np[q])) / k
                )

            overlaps = np.array(overlaps)
            z = standardize(overlaps)

            # ------------------------------------------------------------
            # STEP 5: case study indices
            # ------------------------------------------------------------
            max_idx = int(np.argmax(overlaps))
            min_idx = int(np.argmin(overlaps))
            med_idx = int(np.argsort(overlaps)[len(overlaps) // 2])

            results[(r_i, r_j)] = {
                "overlaps": overlaps,
                "z_scores": z,
                "max_overlap_example": max_idx,
                "min_overlap_example": min_idx,
                "median_overlap_example": med_idx,
            }

    return results, knn_maps

In [ ]:
from datasets import load_dataset


In [ ]:
# def safe_text(s):
#     """
#     Prevents Matplotlib PDF backend crashes caused by:
#     - None
#     - empty strings
#     - null bytes
#     - weird unicode edge cases
#     """
#     if s is None:
#         return " "
#     if not isinstance(s, str):
#         s = str(s)

#     # remove null bytes (CRITICAL for your error)
#     s = s.replace("\x00", "")

#     # collapse empty / whitespace-only strings
#     if s.strip() == "":
#         return " "

#     return s
# # ====================================================================

In [ ]:
def get_doc(dataset, idx):
    if idx is None:
        return None
    ex = dataset[idx]
    # adjust field names depending on your dataset
    return ex["messages"]#.get("text", ex)

In [ ]:
from load_results import abbrev_map

In [ ]:
def align_knn(idx_a, idx_b):
    used = set()
    b_set = set(idx_b)

    aligned = []

    # match in order of idx_a
    for i in idx_a:
        if i in b_set:
            aligned.append(i)
            used.add(i)
        else:
            aligned.append(None)

    # remaining elements from idx_b not used
    backfill = iter([x for x in idx_b if x not in used])

    return [
        v if v is not None else next(backfill, None)
        for v in aligned
    ]

In [ ]:
# results, knn_maps = None, None
# representations = None

In [ ]:
# import matplotlib.font_manager as fm
# import matplotlib.pyplot as plt
# from matplotlib.backends.backend_pdf import PdfPages
# import matplotlib as mpl
# import matplotlib.patches as patches
# import re
# import matplotlib.colors as mcolors
# import os



# fm.fontManager.addfont("fonts/NotoSansCJKtc-Regular.otf")
# fm.fontManager.addfont("fonts/NotoColorEmoji-Regular.ttf")
# fm.fontManager.addfont("fonts/NotoSansEthiopic-VariableFont_wdth,wght.ttf")
# fm.fontManager.addfont("fonts/NotoSansTelugu-VariableFont_wdth,wght.ttf")
# fm.fontManager.addfont("fonts/JetBrainsMono-Regular.ttf")
# fm.fontManager.addfont("fonts/NotoSansDevanagari-VariableFont_wdth,wght.ttf")

# TEXT_FONT_FAMILY = [
#     "Noto Sans CJK TC",
#     "Noto Color Emoji",
#     "Noto Sans Ethiopic",
#     "Noto Sans Telugu",
#     "DejaVu Sans",
#     "Noto Sans Devanagari"
# ]
# CODE_FONT_FAMILY = "JetBrains Mono"

# mpl.rcParams["font.family"] = TEXT_FONT_FAMILY
# mpl.rcParams["mathtext.fontset"] = "dejavusans"

# def darken_color(color, amount=0.6):
#     c = mcolors.to_rgb(color)
#     return tuple(amount * x for x in c)

# def split_text_and_code(text):
#     """
#     Splits text into segments: (segment_text, is_code)
#     Handles ``` code blocks with optional language.
#     """
#     parts = []
#     pattern = re.compile(r"(`{3,}.*?\n)(.*?)(\n`{3,})", re.DOTALL)
#     last = 0

#     for m in pattern.finditer(text):
#         # Text before the code block
#         if m.start() > last:
#             parts.append((text[last:m.start()], False))
#         # Only the inner code
#         code_content = m.group(2)
#         parts.append((code_content, True))
#         last = m.end()

#     # Any remaining text after last code block
#     if last < len(text):
#         parts.append((text[last:], False))

#     return parts
# import re
# import unicodedata

# def safe_text(s):
#     """
#     Fully robust Matplotlib-PDF-safe text cleaner.
#     Fixes:
#     - empty strings
#     - null bytes
#     - zero-width chars
#     - broken unicode
#     - NBSP
#     """

#     if s is None:
#         return " "
#     if not isinstance(s, str):
#         s = str(s)

#     # normalize unicode (VERY important)
#     s = unicodedata.normalize("NFKC", s)

#     # remove null bytes
#     s = s.replace("\x00", "")

#     # remove zero-width characters (CRITICAL for PDF backend crash)
#     s = re.sub(r"[\u200b\u200c\u200d\u2060\ufeff]", "", s)

#     # replace non-breaking space
#     s = s.replace("\xa0", " ")

#     # collapse whitespace-only strings
#     if s.strip() == "":
#         return " "

#     return s


# import matplotlib.font_manager as fm
# import matplotlib.pyplot as plt
# from matplotlib.backends.backend_pdf import PdfPages
# import matplotlib as mpl
# import matplotlib.patches as patches
# import re
# import matplotlib.colors as mcolors
# import os



# fm.fontManager.addfont("fonts/NotoSansCJKtc-Regular.otf")
# fm.fontManager.addfont("fonts/NotoColorEmoji-Regular.ttf")
# fm.fontManager.addfont("fonts/NotoSansEthiopic-VariableFont_wdth,wght.ttf")
# fm.fontManager.addfont("fonts/NotoSansTelugu-VariableFont_wdth,wght.ttf")
# fm.fontManager.addfont("fonts/JetBrainsMono-Regular.ttf")
# fm.fontManager.addfont("fonts/NotoSansDevanagari-VariableFont_wdth,wght.ttf")

# TEXT_FONT_FAMILY = [
#     "Noto Sans CJK TC",
#     "Noto Color Emoji",
#     "Noto Sans Ethiopic",
#     "Noto Sans Telugu",
#     "DejaVu Sans",
#     "Noto Sans Devanagari"
# ]
# CODE_FONT_FAMILY = "JetBrains Mono"

# mpl.rcParams["font.family"] = TEXT_FONT_FAMILY
# mpl.rcParams["mathtext.fontset"] = "dejavusans"

def darken_color(color, amount=0.6):
    c = mcolors.to_rgb(color)
    return tuple(amount * x for x in c)

# def split_text_and_code(text):
#     """
#     Splits text into segments: (segment_text, is_code)
#     Handles ``` code blocks with optional language.
#     """
#     parts = []
#     pattern = re.compile(r"(`{3,}.*?\n)(.*?)(\n`{3,})", re.DOTALL)
#     last = 0

#     for m in pattern.finditer(text):
#         # Text before the code block
#         if m.start() > last:
#             parts.append((text[last:m.start()], False))
#         # Only the inner code
#         code_content = m.group(2)
#         parts.append((code_content, True))
#         last = m.end()

#     # Any remaining text after last code block
#     if last < len(text):
#         parts.append((text[last:], False))

#     return parts
# import re
# import unicodedata

# def safe_text(s):
#     """
#     Fully robust Matplotlib-PDF-safe text cleaner.
#     Fixes:
#     - empty strings
#     - null bytes
#     - zero-width chars
#     - broken unicode
#     - NBSP
#     """

#     if s is None:
#         return " "
#     if not isinstance(s, str):
#         s = str(s)

#     # normalize unicode (VERY important)
#     s = unicodedata.normalize("NFKC", s)

#     # remove null bytes
#     s = s.replace("\x00", "")

#     # remove zero-width characters (CRITICAL for PDF backend crash)
#     s = re.sub(r"[\u200b\u200c\u200d\u2060\ufeff]", "", s)

#     # replace non-breaking space
#     s = s.replace("\xa0", " ")

#     # collapse whitespace-only strings
#     if s.strip() == "":
#         return " "

#     return s
# def draw_chat(
#     ax,
#     idx,
#     messages,
#     x0=0.02,
#     y0=0.95,
#     line_chars=35,
#     fontsize=5,
#     line_chars_cjk_or_cyrillic=10,
#     line_chars_code= 30,
#     onecol=False,
#     add_label=False,
#     color=None,
# ):
#     chat_y = y0

#     def split_lines(text, line_chars, line_chars_cjk_or_cyrillic,is_code,line_chars_code):
#         out = []
#         for line in text.splitlines():
#             has_cjk_or_cyrillic = bool(re.search(r"[\u4e00-\u9fff\u0400-\u04FF]", line))
#             limit = line_chars_cjk_or_cyrillic if has_cjk_or_cyrillic else (line_chars_code if is_code else line_chars)

#             while len(line) > limit:
#                 split_pos = limit if has_cjk_or_cyrillic else max(line.rfind(" ", 0, limit), limit)
#                 out.append(line[:split_pos].rstrip())
#                 line = line[split_pos:].lstrip()
#             if line:
#                 out.append(line)
#         return out

#     pre_splits = []
#     for i, msg in enumerate(messages[:2]):
#         # Determine max_lines
#         max_lines = 3
#         if i == 0:
#             max_lines = 1 if (add_label and onecol) else 3
#         elif i == 1:
#             max_lines = 3 if (add_label and onecol) else 2
#         max_lines = 2*max_lines
#         segments = split_text_and_code(safe_text(msg["content"].replace("Complete the sentence by filling in the blank:\n", "")))


#         norm_segments = []
#         for seg_text, is_code in segments:
#             if not is_code:
#                 seg_text = re.sub(r"\\\((.*?)\\\)", r"$\1$", seg_text, flags=re.DOTALL)
#                 seg_text = re.sub(r"\\\[\s*(.*?)\s*\\\]", r"$\1$", seg_text, flags=re.DOTALL)
#                 seg_text = re.sub(r"^(#{1,3})\s*(.*)$", r"\2", seg_text, flags=re.MULTILINE)
#             norm_segments.append((seg_text.strip("\n"), is_code))


#         split_data = []
#         total_lines = 0
#         for seg_text, is_code in norm_segments:
#             if not seg_text:
#                 continue
#             lines = split_lines(seg_text, line_chars, line_chars_cjk_or_cyrillic,is_code,line_chars_code)
#             if max_lines and total_lines + len(lines) > max_lines:
#                 lines = lines[: max_lines - total_lines]
#                 if lines:
#                     lines[-1] += "..."
#             total_lines += len(lines)
#             split_data.append((lines, is_code))

#         pre_splits.append((i, msg, max_lines, split_data))


#     for i, msg, max_lines, split_data in pre_splits:

#         texts = []
#         cursor_y = chat_y
#         total_height = 0
#         max_width = 0

 
#         for lines, is_code in split_data:
#             wrapped = "\n".join([l.replace("****", "") for l in lines if l.replace("****", "") != ""])
#             # print(wrapped)
#             t = ax.text(
#                 0,
#                 cursor_y,
#                 wrapped,
#                 va="top",
#                 ha="left",
#                 fontsize=fontsize,
#                 color="white",
#                 fontfamily=CODE_FONT_FAMILY if is_code else None,
#             )
#             renderer = ax.figure.canvas.get_renderer()
#             bbox = t.get_window_extent(renderer)
#             h = bbox.height / ax.bbox.height
#             w = bbox.width / ax.bbox.width

#             total_height += h
#             max_width = max(max_width, w)
#             cursor_y -= h + 0.005
#             texts.append(t)

#         # Bubble dimensions
#         bubble_height_axes = total_height + 0.02
#         bubble_width_axes = max_width + 0.02

        
#         base = base_colors[color if color is not None else idx]
#         color_ = base if i % 2 == 0 else  darken_color(base, 0.7)
#         bubble_x = x0 if msg["role"] == "assistant" else 1.0 - bubble_width_axes - 0.02

#         # Draw bubble
#         rect = patches.FancyBboxPatch(
#             (bubble_x, chat_y - bubble_height_axes),
#             bubble_width_axes,
#             bubble_height_axes,
#             boxstyle="round,pad=0.02,rounding_size=0.02",
#             fc=color_,
#             ec=color_,

#             clip_on=False,
#         )
#         ax.add_patch(rect)

     
#         cursor_y = chat_y - 0.005
#         for t in texts:
#             t.set_position((bubble_x + 0.01, cursor_y))
#             renderer = ax.figure.canvas.get_renderer()
#             h = t.get_window_extent(renderer).height / ax.bbox.height
#             cursor_y -= h + 0.005

#         # Optional label
#         if add_label:
#             label = "Completion →" if msg["role"] == "assistant" else "← Prompt"
#             ha = "left" if msg["role"] == "assistant" else "right"
#             lx = (
#                 bubble_x - (0.4 if onecol else 0.3)
#                 if msg["role"] == "assistant"
#                 else bubble_x + bubble_width_axes + (0.3 if onecol else 0.25)
#             )
#             ax.text(
#                 lx,
#                 chat_y - bubble_height_axes / 2,
#                 label,
#                 va="center",
#                 ha=ha,
#                 fontsize=fontsize + (3 if onecol else 5),
#                 fontweight="bold",
#                 transform=ax.transAxes,
#             )

#         chat_y -= bubble_height_axes + 0.06

#     return chat_y

# def plot_chat_case_study(example_to_explain, docs, pdf_path="figures/chat_case_study_fixed.pdf",onecol=False, underline=None, color=None):
    
#     C= 3 if onecol else 5
#     page_width_pt = 219.08614 if onecol else 455.24411
#     page_height_pt = 400 if onecol else 704.60031 -198.425
#     page_width_in = page_width_pt / 72
#     page_height_in = page_height_pt / 72

#     fig = plt.figure(figsize=(page_width_in, page_height_in))
#     fig.canvas.draw() 
#     fig.set_constrained_layout(False)
#     fig.set_tight_layout(False)
#     fig.subplots_adjust(left=0.1, right=1, top=1, bottom=0)


#     margin_left = 0
#     col_spacing = 0.00
#     col_width = (1 / C) * (1 - margin_left) * 0.98
#     grid_width = C * col_width + (C - 1) * col_spacing
#     grid_center_x = margin_left + grid_width / 2
    
#     row_height = 0.12#0.12
#     row_spacing = 0.00

    
#     ax1_width = 0.582
#     ax1_left = grid_center_x - ax1_width / 2

#     ax1 = fig.add_axes([ax1_left, 0.75, ax1_width, 0.25])       
    
#     chat_end =draw_chat(ax1, 0, example_to_explain, 
#                         x0=0.05, 
#                         y0=0.97,  
#                         line_chars=58 if onecol else 78,
#                         line_chars_code=40 if onecol else 57,
#                         fontsize=5.5, add_label=True, onecol=onecol, color=0)


#     ax1.set_xticks([])
#     ax1.set_yticks([]) 
#     ax1.set_frame_on(False)
#     ax1_bottom_fig = ax1.get_position().y0 


#     n_rows = len(docs)

#     start_x = margin_left
#     start_y = ax1_bottom_fig + 0.07 if onecol else ax1_bottom_fig + 0.00  # small gap

#     docs = [(method_name, docs_col) for method_name, docs_col, in docs if "FL λ=0" != method_name]
    
    

#     # pred_gains = [m[-3] for m in methods]
#     # min_gain = min(pred_gains)
#     # max_gain = max(pred_gains)
#     merged_positions = set()
#     for r, (method_name, docs_col) in enumerate(docs): 
#         n_cols = min(len(docs_col), C)
#         y = start_y - r * (row_height + row_spacing)
#         for c, doc in enumerate(docs_col[0:C]):

#             # skip cells already consumed by a merge from row above
#             if (r, c) in merged_positions:
#                 continue

     
#             x = start_x + c * (col_width + col_spacing)
#             should_merge = (
#                 r + 1 < len(docs)
#                 and c < len(docs[r+1][1])
#                 and docs[r+1][1][c] is None
#             )

#             current_height = row_height

#             if should_merge:
#                 current_height = row_height * 2 + row_spacing
#                 merged_positions.add((r + 1, c))  # mark next row as consumed

#             # shift downward so merged box covers both rows
#             y_adjusted = y - (current_height - row_height)

#             ax_outer = fig.add_axes([
#                 x,
#                 y_adjusted,
#                 col_width,
#                 current_height
#             ])
#             ax_outer.set_xlim(0, 1)
#             ax_outer.set_ylim(0, 1)
#             ax_outer.set_aspect("auto")
#             ax_outer.set_autoscale_on(False)
#             ax_outer.autoscale(False)
#             pad = 0.04
#             ax = ax_outer.inset_axes([pad, pad, 1 - 2 * pad, 1 - 2 * pad])
#             ax.set_xlim(0, 1)
#             ax.set_ylim(0, 1)
#             ax.set_aspect("auto")
#             ax.autoscale(False)
#             ax.set_autoscale_on(False)

            
#             is_left_edge = (c == 0)
#             is_right_edge = (c == n_cols - 1)
#             is_top_edge = (r == 0)
#             is_bottom_edge = (r == n_rows - 1)

   
#             ax_outer.spines['left'].set_visible(not is_left_edge)
#             ax_outer.spines['right'].set_visible(not is_right_edge)
#             ax_outer.spines['top'].set_visible(not is_top_edge)
#             ax_outer.spines['bottom'].set_visible((not is_bottom_edge) and (doc is None))
#             for spine_name in ['left', 'right', 'top', 'bottom']:
#                 spine = ax_outer.spines[spine_name]
#                 if spine.get_visible():
#                     spine.set_edgecolor("#DDDDDD")  
#                     spine.set_linewidth(0.8)
#             ax_outer.set_xticks([]); ax_outer.set_yticks([])
#             ax.set_xticks([]); ax.set_yticks([])
#             ax.set_frame_on(False)
            
#             method_name_display = method_name#.replace("naive Top-5",)# "naive Top-5 (FL λ=0)")
#             if doc is not None:
#                 should_merge = (r+1 < len(docs)) and (docs[r+1][1][c] is None)
#                 # print("should merge", should_merge)
#                 chat_end = draw_chat(ax, r+1,doc, 
#                                     x0=0.05 , 
#                                     y0=0.95 if not should_merge else 0.7,
#                                     line_chars=24  if onecol else 29, 
#                                     line_chars_code=22  if onecol else 27,
#                                     line_chars_cjk_or_cyrillic=15, 
#                                     fontsize=4.5,
#                                     onecol=onecol,
#                                     color=color)
#             ax.set_xticks([]); ax.set_yticks([])
#             # print(pred_gain, max_gain, min_gain, underline)
#             # mark = (pred_gain == max_gain and underline == "max") or (pred_gain == min_gain and underline == "min")
#             # # print(mark)
#             # if mark:
#             #     ax_outer.set_facecolor("#cdcdcd")

#             if r == 0:
#                 pass
                
#             # draw row label once per row
#         visible_cols = [
#             c for c in range(min(len(docs_col), C))
#             if (r, c) not in merged_positions
#         ]

#         if visible_cols:
#             first_c = visible_cols[0]
#             x_label = start_x + first_c * (col_width + col_spacing)

#             # if first visible cell merged downward, center label on merged height
#             merged_down = (
#                 r + 1 < len(docs)
#                 and first_c < len(docs[r+1][1])
#                 and docs[r+1][1][first_c] is None
#             )

#             label_height = row_height * 2 if merged_down else row_height
#             label_y = y - (label_height - row_height) + label_height / 2
#             # print("label_y",label_y)
#             fig.text(
#                 0,
#                 0.71 if r == 1 else  0.8300000000000001,
#                 method_name_display.replace("Top-5", "").replace("FL", ""),
#                 fontsize=7,
#                 rotation=90,
#                 va="center",
#                 ha="right",
#                 fontweight="normal"
#             )
#     print(pdf_path)
#     os.makedirs(os.path.dirname(pdf_path), exist_ok=True)
#     with PdfPages(pdf_path) as pdf:
#         pdf.savefig(fig, bbox_inches="tight", pad_inches=0.02)

#     plt.show()


#     plt.close(fig)

In [ ]:
import numpy as np
import matplotlib.patches as patches
from matplotlib.image import BboxImage
from matplotlib.transforms import Bbox

In [ ]:
import textwrap

In [ ]:
representations = None
results,knn_maps = None, None

In [ ]:

from IPython.display import display, HTML
import html
import re
import markdown


# -----------------------------
# TEXT PREPROCESSING
# -----------------------------
def preprocess_latex(text):
    # \[ ... \] → $$ ... $$
    text = re.sub(r"\\\[\s*", r"$$", text)
    text = re.sub(r"\s*\\\]", r"$$", text)

    # \( ... \) → $ ... $
    text = re.sub(r"\\\((.*?)\\\)", r"$\1$", text)

    return text
# -----------------------------
# MESSAGE RENDERING (Markdown + LaTeX)
# -----------------------------
def render_message(msg):
    role = msg["role"]
    content = preprocess_latex(msg["content"])

    # Markdown → HTML (includes code blocks)
    content_html = markdown.markdown(
        content,
        extensions=["fenced_code", "tables"]
    )

    side = "left" if role == "assistant" else "right"
    bubble_class = "assistant" if role == "assistant" else "user"

    return f"""
    <div class="msg-row {side}">
        <div class="bubble {bubble_class}">
            {content_html}
        </div>
    </div>
    """


def render_chat(messages, top=False):
    if top:
        return "<div class='mini-chat' id='top'>" + "".join(
        render_message(m) for m in messages[:2]
    ) + "</div>"
    else:
        return "<div class='mini-chat'>" + "".join(
            render_message(m) for m in messages[:2]
        ) + "</div>"



In [ ]:
markdown.markdown(
        f"""def is_palindrome(string):\n    # Remove spaces, punctuation, and convert to lowercase\n    cleaned_string = ''.join(char.lower() for char in string if char.isalnum())\n    \n    # Check if the cleaned string is equal to its reverse\n    return cleaned_string == cleaned_string[::-1]""",
        extensions=["fenced_code", "tables"]
    )

In [ ]:
def plot_chat_case_study_html(
    example_to_explain,
    docs,
    pdf_path=None,
    onecol=False,
    color=None,
    color_top=None,
    C=None,
    max_chat_height=6*6,
):

    if C is None:
        C = 3 if onecol else 5

    docs = [
        (method_name, docs_col)
        for method_name, docs_col in docs
        if method_name != "FL λ=0"
    ]

    top_chat_html = render_chat(example_to_explain,top=True)

    grid_rows_html = ""

    merged_positions = set()

    # -----------------------------
    # BUILD TABLE ROWS
    # -----------------------------
    for r, (method_name, docs_col) in enumerate(docs):

        row_html = ""

        # row label cell (rotated)
        label_cell = f"""
        <td class="row-label">
            {method_name.replace("Top-5", "").replace("FL", "")}
        </td>
        """

        row_html += label_cell

        for c, doc in enumerate(docs_col[:C]):

            if (r, c) in merged_positions:
                continue

            if doc is None:
                continue

            rowspan = 1

            should_merge = (
                r + 1 < len(docs)
                and c < len(docs[r + 1][1])
                and docs[r + 1][1][c] is None
            )

            if should_merge:
                rowspan = 2
                merged_positions.add((r + 1, c))

            row_html += f"""
            <td class="cell" rowspan="{rowspan}">
                {render_chat(doc)}
            </td>
            """

        grid_rows_html += f"<tr>{row_html}</tr>"

    # -----------------------------
    # FINAL HTML
    # -----------------------------
    html_output = f"""
        <style>
        .container {{
        font-size: 2.5pt;
    }}

    /* force inheritance everywhere */
    .container * {{
        font-size: inherit;
    }}
        .container {{
            width: 219.08614pt;
            background-color: white;
            margin: auto;
            font-family: -apple-system, BlinkMacSystemFont, sans-serif;
        }}

        /* ---------------- TABLE ---------------- */
        .layout-table {{
            width: 100%;
            border-collapse: separate;
            border-spacing: 1px;
            table-layout: fixed;  
        }}

        /* ---------------- TOP CHAT ---------------- */
        .top-chat {{
            width: 50%;
            margin: 0 auto 25px auto;
            padding: 16px;
            border: 1px solid #ddd;
            border-radius: 14px;
        }}

        .top-chat-cell {{
            padding: 2px;
            border: 1px solid #ddd;
            background: white;
        }}

        /* ---------------- ROW LABEL ---------------- */
        .row-label {{
           
            font-size: 4px;
            font-weight: 500;
            color: #444;
         
            vertical-align: middle;
            transform: rotate(-90deg);
            white-space: nowrap;
        }}

        /* ---------------- CELL ---------------- */
        .cell {{
            border: 1px solid #ddd;
            padding: 1px;
            background: #fafafa;
            vertical-align: top;
            max-height: 5em;
            overflow: hidden;
            vertical-align: middle;
        }}

        /* ---------------- CHAT ---------------- */
        .mini-chat {{
            overflow: hidden;
        }}


        /* ---------------- BUBBLES ---------------- */
        .bubble {{
            display: block;
            max-width: 100%;
            padding: 4px;
            border-radius: 4px;
          
  
            overflow: hidden;
            max-height: {max_chat_height}px;
            position: relative;
        }}

        .assistant {{
            background: #E5E5EA;
            color: black;
            margin-right: 10%;
        }}

        .user {{
            background: {color};
            color: white;
            margin-left: 10%;
            
        }}
        #top   .bubble.user {{
            background: {color_top};
        }}

        .bubble::after {{
               content: "";
    position: absolute;
    left: 0;
    right: 0;
    bottom: 0;
    height: 24px;
    pointer-events: none;
    opacity: 0;
}}

.bubble.is-overflowing::after {{
    opacity: 1;
}}

.assistant.is-overflowing::after {{
    background: linear-gradient(to bottom,
        rgba(229,229,234,0),
        rgba(229,229,234,1));
}}

.user.is-overflowing::after {{
    background: linear-gradient(to bottom,
        rgba(0,122,255,0),
        {color});
}}

        .assistant::after {{
            background: linear-gradient(to bottom,
                rgba(229,229,234,0),
                rgba(229,229,234,1));
        }}

        .user::after {{
            background: linear-gradient(to bottom,
                rgba(0,122,255,0),
                {color});
        }}
        #top .user::after {{
            background: linear-gradient(to bottom,
                rgba(0,122,255,0),
                {color_top});
        }}


        /* ---------------- CODE ---------------- */
        pre {{
            
            padding: 8px;
            border-radius: 10px;
            overflow-x: auto;

        }}

        code {{
            font-family: Menlo, Monaco, monospace;
        }}
        
        p {{
        margin: 0;
        padding: 0;
    }}
    
    ul, ol {{
    margin: 0;
    padding: 0;
    list-style-position: inside;
}}

li {{
    margin: 0;
    padding: 0;
    line-height: 1.15;
}}

li p {{
    margin: 0;
    padding: 0;
}}
    </style>

    <!-- MathJax -->
    <script>
    window.MathJax = {{
      tex: {{
        inlineMath: [['$', '$'], ['\\\\(', '\\\\)']]
      }},
      startup: {{
    ready() {{

      MathJax._.core.MmlTree.MmlNodes.math.MmlMath.defaults.scriptminsize = '4pt';

      // continue normal startup
      MathJax.startup.defaultReady();
    }}
      }}
    }};
    </script>
    
    <script src="https://cdn.jsdelivr.net/npm/mathjax@3/es5/tex-chtml.js"></script>

    <script>
    window.onload = function () {{
        if (window.MathJax) {{
            MathJax.typeset();
        }}
        
        const MAX_HEIGHT = {max_chat_height}; // inject Python value

    document.querySelectorAll('.bubble').forEach(el => {{
        const contentHeight = el.scrollHeight;

        if (contentHeight > MAX_HEIGHT) {{
            el.classList.add('is-overflowing');
        }}
    }});
    
    }};
    </script>

    <div class="container">

        <table class="layout-table">
        <colgroup>
        <col style="width: 10px;">
        <col style="width: calc((100% - 10px) / {C});" span="{C}">
    </colgroup>

            <!-- MERGED FIRST ROW (TOP CHAT) -->
            <tr>
                <td colspan="{C+1}" class="top-chat-cell">
                    {top_chat_html}
                </td>
            </tr>

            <!-- GRID ROWS -->
            {grid_rows_html}

        </table>

    </div>
    """

    if pdf_path:
        import pdfkit

     
        html_path = pdf_path.replace(".pdf", ".html")
        with open(html_path, "w", encoding="utf-8") as f:
            f.write(html_output)


        # options = {
        #     "margin-top": "0",
        #     "margin-bottom": "0",
        #     "margin-left": "0",
        #     "margin-right": "0",
        #     "encoding": "UTF-8",
        #     "no-outline": None,
        #     "enable-local-file-access": None,
        #     "disable-smart-shrinking": None
        # }

        # pdfkit.from_string(html_output, pdf_path, options=options)

        print(f"Saved HTML to {html_path}")
        print(f"Saved PDF to {pdf_path}")

In [ ]:
import matplotlib.pyplot as plt
import os
from matplotlib.backends.backend_pdf import PdfPages



In [ ]:
# import matplotlib.pyplot as plt
# import os
# from matplotlib.backends.backend_pdf import PdfPages


# def plot_chat_case_study(example_to_explain, docs,
#                           pdf_path="figures/chat_case_study_fixed.pdf",
#                           onecol=False,
#                           underline=None,
#                           color=None):

#     # -------------------------
#     # FIGURE (UNCHANGED)
#     # -------------------------
#     C = 3 if onecol else 5
#     R = 2

#     fig = plt.figure(figsize=(6, 8))
#     fig.set_constrained_layout(False)
#     fig.set_tight_layout(False)

#     # -------------------------
#     # CHAT (TOP) — EXACT SAME
#     # -------------------------
#     ax1 = fig.add_axes([0.2, 0.72, 0.6, 0.25])

#     ax1.set_xlim(0, 1)
#     ax1.set_ylim(0, 1)
#     ax1.set_aspect("auto")
#     ax1.set_autoscale_on(False)

#     draw_chat(
#         ax1,
#         example_to_explain,
#         x0=0.05,
#         y0=0.97,
#         x1=1,
#         y1=0,
#         fontsize=5.5,
#         color=0
#     )

#     # -------------------------
#     # GRID (UNCHANGED TEMPLATE)
#     # -------------------------
#     gap_between = 0.03
#     grid_top = 0.72 - gap_between
#     grid_bottom = 0
#     grid_height = grid_top - grid_bottom

#     col_spacing = 0.01
#     col_w = (1 - col_spacing * (C - 1)) / C

#     gap = 0.01
#     usable_height = grid_height - gap * (R - 1)
#     row_h = usable_height / R

#     # -------------------------
#     # GRID DRAW (RESTORED STRUCTURE)
#     # -------------------------
#     merged_positions = set()


#     for r, (method_name, docs_col) in enumerate(docs):
#         for c, doc in enumerate(docs_col):
#             if doc is None:
#                 continue
            
#             should_merge = (
#                 r + 1 < len(docs)
#                 and c < len(docs[r + 1][1])
#                 and docs[r + 1][1][c] is None
#             )

#             left = c * (col_w + col_spacing)

#             row_unit = row_h + gap

#             bottom = grid_bottom + (R - 1 - r) * row_unit

#             height = row_h

#             if should_merge:
#                 bottom -= (row_h + gap)   # move start down one row
#                 height = row_h * 2 + gap  # span two rows

#             ax = fig.add_axes([left, bottom, col_w, height])
#             ax.set_xlim(0, 1)
#             ax.set_ylim(0, 1)
#             ax.set_aspect("auto")
#             ax.set_xticks([])
#             ax.set_yticks([])
#             ax.set_frame_on(True)

#             # -------------------------
#             # RESTORE ORIGINAL EXPECTATION
#             # docs = (method_name, docs_col)
#             # -------------------------
    
     
            
            
#             draw_chat(
#                 ax,
#                 doc,
#                 x0=0.05,
#                 y0=0.95,
#                 x1=1,
#                 y1=0,
#                 fontsize=4.5,
#                 color=color
#             )
            
#         # break
#     # -------------------------
#     # SAVE
#     # -------------------------
#     os.makedirs(os.path.dirname(pdf_path), exist_ok=True)
#     with PdfPages(pdf_path) as pdf:
#         pdf.savefig(fig, bbox_inches="tight", pad_inches=0.02)

#     plt.show()
#     plt.close(fig)

In [ ]:
data

In [ ]:
import shutil

for model_name, representation_data in data.items():
    shutil.rmtree("/tmp/cache/representations", ignore_errors=True) 
    if not DEBUG or (DEBUG and (representations is None)):
        representations = [(r, args_, d_fn()) for r, args_, d_fn in representation_data]
    # print(representations)
    # break
    # extract metadata
    namespace = representations[0][1]
    dataset_name = namespace.train_dataset_name
    dataset_split = namespace.train_dataset_split


    dataset = load_dataset(dataset_name, split=dataset_split)
    # print("representations",representations)
    if not DEBUG or (DEBUG and (results is None)):
        results,knn_maps = analyze_representations(representations, dataset, k=5)

    for pair, stats in results.items():
        r_i, r_j = pair

        print("\n==============================")
        print(pair)

        def process(mode, idx_key):
            print(mode)
            idx = stats[idx_key]

            idx_a = knn_maps[r_i][idx].tolist()
            idx_b = knn_maps[r_j][idx].tolist()

            aligned_b = align_knn(idx_a, idx_b)

            idx_a_sorted, idx_b_sorted = zip(*[(g[0],(g[1] if (g[0] != g[1]) else None)) for g in
                sorted(
                    zip(idx_a, aligned_b),
                    key=lambda x: x[0] == x[1],
                    reverse=True
                )
            ])
            

            return idx, idx_a_sorted, idx_b_sorted

        filename = lambda mode: f"./figures/case_study/{os.path.basename(model_name).replace('_','-')}/{r_i.split('.')[-1]}_{os.path.basename(r_j.split('.')[-1])}_{mode}.pdf"
        
        max_file = filename("max")
        min_file = filename("min")
        # ---------- MAX ----------
        max_idx, idx_a_max_sorted, idx_b_max_sorted = process("max", "max_overlap_example")
        os.makedirs(os.path.dirname(max_file), exist_ok=True)
        plot_chat_case_study_html(
            get_doc(dataset, max_idx),
            [
                (abbrev_map[pair[0]], [get_doc(dataset, i) for i in idx_a_max_sorted]),
                (abbrev_map[pair[1]], [get_doc(dataset, i) for i in idx_b_max_sorted]),
            ],
            pdf_path=max_file,
            # onecol=False,
            color_top=mcolors.to_hex(base_colors[0]),
            color=mcolors.to_hex(darken_color(base_colors[1], 1))
        )


        # ---------- MIN ----------
        min_idx, idx_a_min_sorted, idx_b_min_sorted = process("min", "min_overlap_example")

        plot_chat_case_study_html(
            get_doc(dataset, min_idx),
            [
                (abbrev_map[pair[0]], [get_doc(dataset, i) for i in idx_a_min_sorted]),
                (abbrev_map[pair[1]], [get_doc(dataset, i) for i in idx_b_min_sorted]),
            ],
            pdf_path=min_file,
            color_top=mcolors.to_hex(base_colors[0]),
            color=mcolors.to_hex(darken_color(base_colors[2], 1))
        )

        latex = f"""
    \\begin{{figure*}}
        \\centering

        \\begin{{minipage}}{{1\\textwidth}}
            \\centering
            \\includegraphics[width=\\linewidth]{{{max_file}}}
        \\end{{minipage}}
        

        \\begin{{minipage}}{{1\\textwidth}}
            \\centering
            \\includegraphics[width=\\linewidth]{{{min_file}}}
            \\caption{{Examples with maximum (top) and minimum (bottom) 5-nearest neighbor overlap between {abbrev_map[r_i]} and {abbrev_map[r_j]} for {model_name.replace('_', ' ')}.}}
        \\end{{minipage}}

    \\end{{figure*}}
    """

        print(latex)
        